# 第8–11章网页适配验证
教材 pp.102–145；公式复用 service，验证覆盖全部14个注册操作。示例均为构造数据。

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "service/catalog.py").exists())
sys.path.insert(0, str(root))
import json
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from controller.chapter_groups.dependence import operations, examples, example_frame, calculate

def run(operation, frame=None, **parameters):
    definition = next(o for o in operations() if o['id'] == operation)
    example, metadata = example_frame(definition['example_id'])
    return calculate(example if frame is None else frame, metadata, {'operation':operation,'parameters':parameters})

def table(result, name):
    return pd.DataFrame(next(t['rows'] for t in result['tables'] if t['name']==name))

def metric(result, name):
    return next(m['value'] for m in result['metrics'] if m['name']==name)


In [ ]:
# 每个操作都运行自己的默认示例；输入不得修改，结果必须严格JSON兼容。
for operation in operations():
    frame, metadata = example_frame(operation['example_id'])
    before = frame.copy(deep=True)
    result = calculate(frame, metadata, {'operation':operation['id'], 'parameters':{}})
    pd.testing.assert_frame_equal(frame, before)
    assert any(result[key] for key in ['metrics','tables','charts'])
    json.dumps(result, allow_nan=False)
    print(operation['id'], 'OK')
assert {o['chapter'] for o in operations()} == {8,9,10,11}
for method in ("spearman", "kendall"):
    json.dumps(run("ch8_moments", method=method), allow_nan=False)


## 第8章：相关、组合矩与联合违约（pp.104–114）

In [ ]:
data = pd.DataFrame({'a':[-.1,0,.1],'b':[-.2,0,.2]})
result = run('ch8_moments', data, weights=[.5,.5])
assert_allclose(metric(result,'mean'),0,atol=1e-15)
assert_allclose(metric(result,'volatility'),.15)
assert_allclose(table(result,'相关矩阵')[['a','b']],1)
positions = pd.DataFrame({'amount':[100.,0.],'sigma':[.2,.3], 'weight':[.5,.5]})
result = run('ch8_covariance_var',positions,correlation=[[1,0],[0,1]],horizon=1,confidence_level=.5)
# 中位数的零均值正态VaR等于零，协方差为对角矩阵。
assert_allclose(metric(result,'VaR（金额）'),0)
assert_allclose(table(result,'协方差矩阵')[['0','1']],np.diag([.04,.09]))
result = run('ch8_implied',pd.DataFrame({'weight':[.5,.5],'sigma':[.2,.2]}),portfolio_volatility=np.sqrt(.03))
assert_allclose(metric(result,'隐含平均相关系数'),.5)
result = run('ch8_joint_default',pd.DataFrame({'probability':[.02,.03]}))
assert_allclose(metric(result,'联合违约概率'),.0006)


## 第9章：输入顺序、状态相关、离散均值回归（pp.122–126）

In [ ]:
data = pd.DataFrame({'a':np.arange(20.),'b':2*np.arange(20.)})
result = run('ch9_rolling',data,window=3)
rolling=table(result,'滚动相关').correlation
assert rolling.iloc[:2].isna().all()
assert_allclose(rolling.iloc[2:],1)
data=pd.DataFrame({'a':[1.,2.,3.,1.,2.,3.], 'b':[1.,2.,3.,3.,2.,1.], 'regime':['A']*3+['B']*3})
result=run('ch9_regimes',data)
regimes=table(result,'状态相关矩阵').set_index(['regime','variable'])
assert_allclose(regimes.loc[('A','a'),'b'],1)
assert_allclose(regimes.loc[('B','a'),'b'],-1)
x=[80.]
for _ in range(20): x.append(x[-1]+.3*(100-x[-1]))
result=run('ch9_reversion',pd.DataFrame({'level':x}),lags=[1,2])
assert_allclose(metric(result,'speed'),.3)
assert_allclose(metric(result,'long_run_mean'),100)


## 第10章：确定性种子、独立联合概率与逆CDF（pp.130–133）

In [ ]:
first=run('ch10_sample',seed=19,n_samples=200)
second=run('ch10_sample',seed=19,n_samples=200)
assert first==second
result=run('ch10_joint',pd.DataFrame({'probability':[.0651,.2383]}),correlation=[[1,0],[0,1]])
assert_allclose(metric(result,'联合违约概率'),.0651*.2383)
assert_allclose(metric(run('ch10_joint'),'联合违约概率'),.0344,atol=.0001)
uniforms=pd.DataFrame({'a':[.1,.5,.9], 'b':[.2,.6,.8]})
times=table(run('ch10_default_times',uniforms,hazards=[.02,.05]),'映射结果')
assert_allclose(1-np.exp(-times.to_numpy()*[.02,.05]),uniforms)
from scipy.stats import norm,expon,t
mapped=table(run('ch10_marginals',uniforms),'映射结果')
assert_allclose(norm.cdf(mapped.a),uniforms.a)
assert_allclose(expon.cdf(mapped.b,scale=2),uniforms.b)
mapped=table(run('ch10_marginals',uniforms,marginals=[{'type':'student_t','df':4},{'type':'normal','loc':2,'scale':3}]),'映射结果')
assert_allclose(t.cdf(mapped.a,df=4),uniforms.a)
assert_allclose(norm.cdf(mapped.b,loc=2,scale=3),uniforms.b)


## 第11章：回归系数、PCA重构与敞口抵消（pp.136–145）

In [ ]:
x=np.arange(10.)
data=pd.DataFrame({'a':x,'target':.06+.842*x})
result=run('ch11_regression',data,predictor_columns=['a'],hedge_dv01=[50])
assert_allclose(table(result,'回归摘要').coefficient,[.06,.842])
assert_allclose(table(result,'对冲数量').value,[-1.684])
assert_allclose(table(result,'拟合与残差').residual,0,atol=1e-12)
frame,_=example_frame('ch11_hedge')
result=run('ch11_pca',frame,n_components=3)
assert_allclose(metric(result,'保留解释方差比例'),1)
loadings=table(result,'一标准差载荷').set_index('variable').to_numpy()
assert_allclose(loadings@loadings.T,frame[['a','b','target']].cov(),atol=1e-12)
result=run('ch11_factor_hedge')
assert_allclose(metric(result,'最大绝对残余因子敞口'),0,atol=1e-12)
positions=table(result,'工具对冲数量').value.to_numpy()
assert_allclose(np.array([[2,1],[1,3]]).T@positions+[10,20],0,atol=1e-12)


## 调用边界：拒绝非法输入，不做隐式修复

In [ ]:
invalid = [
    ('ch10_joint',None,{'correlation':[[1,2],[2,1]]}),
    ('ch10_default_times',pd.DataFrame({'a':[0.], 'b':[.5]}),{}),
    ('ch11_regression',pd.DataFrame({'a':[1.,2.,3.,4.],'b':[2.,4.,6.,8.],'target':[1.,2.,3.,4.]}),{}),
    ('ch8_joint_default',pd.DataFrame({'probability':[.01,.99]}),{'correlation':1}),
    ('ch9_rolling',None,{'window':2.5}),
    ('ch10_sample',None,{'n_samples':20001}),
    ('ch8_moments',pd.DataFrame({'a':[1.,np.nan],'b':[2.,3.]}),{}),
]
for operation,frame,parameters in invalid:
    try: run(operation,frame,**parameters)
    except ValueError: pass
    else: raise AssertionError(f'{operation} accepted invalid input')
print('全部14操作及边界验证通过')
